In [1]:
# Cell 1: Imports and Setup
import mne
import numpy as np
import matplotlib.pyplot as plt
from mne.datasets import eegbci
from mne.io import concatenate_raws, read_raw_edf
import  colorama

# Ensure matplotlib plots display inline in the notebook
%matplotlib inline

# Set MNE logging level to 'WARNING' to reduce text output clutter in the notebook
mne.set_log_level('WARNING')


# Downloading and Parsing the Data

In [28]:
# Cell 2: Downloading and Parsing the Data

# Target all 109 subjects available in the dataset
# Note: To test quickly, change this to range(1, 3)
# subjects = list(range(1, 110))
subjects = list(range(1, 3))

#! Define the runs you want to analyze.
# Ligne de base (Baseline), yeux ouverts
run_open_eyes = [1] 
# Ligne de base (Baseline), yeux fermés
run_closed_eyes = [2] 
# Motor execution: Open and close fist (Left vs. Right)
run_execution_hand = [3, 7, 11]
# Motor imagery: Imagine opening and closing fist (Left vs. Right)
run_imagery_hand = [4, 8, 12]
# Motor execution: Open and close both fists vs. both feet
run_execution_both_hands_feet = [5, 9, 13]
# Motor imagery: Imagine opening and closing both fists vs. both feet
run_imagery_both_hands_feet = [6, 10, 14]

# runs the entire set of runs for the BCI Competition IV dataset, which includes baseline, motor execution, and motor imagery tasks
runs = (
    run_open_eyes
    + run_closed_eyes
    + run_execution_hand
    + run_imagery_hand
    + run_execution_both_hands_feet
    + run_imagery_both_hands_feet
)


# Define your local data path (this should match your .gitignore)
data_path = './mne_data'

print(f"Initiating download for {len(subjects)} subjects. This may take a while...")

# Initialize an empty list to hold our raw data objects
all_raws = []

for subject in subjects:
    try:
        # Download data for the specific subject and runs
        raw_fnames = eegbci.load_data(subject, runs, path=data_path)
        
        # Parse the downloaded EDF files into MNE Raw objects
        subject_raws = [read_raw_edf(f, preload=True) for f in raw_fnames]
        
        # Concatenate the runs for this subject into a single continuous object
        subject_raw = concatenate_raws(subject_raws)
        
        # Standardize channel names to the 10-20 system
        eegbci.standardize(subject_raw)
        
        # Append to our master list
        all_raws.append(subject_raw)
        
    except Exception as e:
        print(f"Could not load data for subject {subject}: {e}")

# If you want to merge all subjects into one massive continuous recording (use with caution regarding RAM):
# full_dataset = concatenate_raws(all_raws)

print(f"Successfully loaded data for {len(all_raws)} subjects.")

Initiating download for 2 subjects. This may take a while...
Successfully loaded data for 2 subjects.


# Cell 3: Data Parsing, Validation & Event Extraction


In [29]:
all_raws[0]

<RawEDF | S001R01.edf, 64 x 259520 (1622.0 s), ~126.8 MiB, data loaded>

In [30]:
len(all_raws) # Display the number of subjects successfully loaded

2

In [ ]:


# For demonstration, we will process the first subject's raw data from our 'all_raws' list
raw = all_raws[0]


In [ ]:
all_raws[0].info.keys()

dict_keys(['acq_pars', 'acq_stim', 'ctf_head_t', 'description', 'dev_head_t', 'dev_ctf_t', 'dig', 'experimenter', 'utc_offset', 'device_info', 'file_id', 'highpass', 'hpi_subsystem', 'kit_system_id', 'helium_info', 'line_freq', 'lowpass', 'meas_date', 'meas_id', 'proj_id', 'proj_name', 'subject_info', 'xplotter_layout', 'gantry_angle', 'bads', 'chs', 'comps', 'events', 'hpi_meas', 'hpi_results', 'projs', 'proc_history', 'custom_ref_applied', 'sfreq', 'ch_names', 'nchan'])

In [3]:
# ==========================================
# !1. Validate Channel Count & Sampling Rate
# ==========================================
print(
    colorama.Style.BRIGHT
    + colorama.Fore.GREEN
    + "--- Data Validation ---"
    + colorama.Style.RESET_ALL
)

for index, subject_raw in enumerate(all_raws):

    # nchan is the number of channels in the EEG (Electroencephalography) data
    n_channels = subject_raw.info["nchan"]
    # sfreq is the sampling frequency of the EEG data, which indicates how many samples per second were recorded
    sfreq = subject_raw.info["sfreq"]

    # Check for the expected 160 Hz rate and 64 channels
    # the frequency of 160 hz is a common sampling rate for EEG data
    if sfreq != 160.0:
        print(f"{colorama.Fore.RED}Subject {index + 1}:")
        print(f"Sampling Frequency: {sfreq} Hz")
        print(
            "⚠️ WARNING: Sampling rate is not 160 Hz. Consider excluding or resampling this subject."
        )

    # channel should be 64 for the BCI Competition IV dataset, which is a common standard for EEG datasets
    # one channel per electrode is used to capture the electrical activity of the brain, and 64 covers a wide area of the scalp,
    # providing a good balance between spatial resolution and computational efficiency
    if n_channels != 64:
        print(f"{colorama.Fore.RED}Subject {index + 1}:")
        print(f"Number of Channels: {n_channels}")
        print("⚠️ WARNING: Channel count is not 64. Check the dataset integrity.")

print(
    f"{colorama.Fore.GREEN}✅ for all the rest subjects, the sampling rate is exactly 160 Hz and the channel count is 64, which are the expected values for this dataset."
)

--- Data Validation ---
✅ for all the rest subjects, the sampling rate is exactly 160 Hz and the channel count is 64, which are the expected values for this dataset.



# 2. Extract and Map Event Codes (labels targets)

In [31]:
# ==========================================
# !2. Extract and Map Event Codes
# ==========================================
# The EDF+ format stores events as string annotations ('T0', 'T1', 'T2').
# We must convert these strings into a numeric matrix (events array) for scikit-learn.

# Define the custom mapping explicitly based on your requirements
custom_mapping = {
    "T0": 0,  # Rest
    "T1": 1,  # Left fist motion/imagery
    "T2": 2,  # Right fist motion/imagery
}

# Extract the events using the custom mapping
# ? events is a numpy array where each row corresponds to an event, and the columns represent:
# ?  the sample (index), (previous event - default to 0), (event code).
# ? Event Dictionary Mapping: This confirms that your custom_mapping worked. I
events, event_dict = mne.events_from_annotations(all_raws[0], event_id=custom_mapping)

print("\n--- Event Extraction ---")
print(f"Total events found: {len(events)}")
print("Event Dictionary Mapping:", event_dict)

# Quick sanity check to see the first 5 events (Timestamp, duration, event_code)
print("\nFirst 5 event triggers (Sample Index, Previous Event, Event Code):")
print(events[:5])


--- Event Extraction ---
Total events found: 362
Event Dictionary Mapping: {np.str_('T0'): 0, np.str_('T1'): 1, np.str_('T2'): 2}

First 5 event triggers (Sample Index, Previous Event, Event Code):
[[    0     0     0]
 [ 9760     0     0]
 [19520     0     0]
 [20192     0     2]
 [20848     0     0]]


In [52]:
# Cell 3: Batch Validation & Event Extraction for All Subjects

# Define the custom mapping explicitly based on project requirements
custom_mapping = {
    "T0": 0,  # Rest
    "T1": 1,  # Left fist motion/imagery
    "T2": 2,  # Right fist motion/imagery
}

# Initialize lists to store our validated data
valid_raws = []
all_events = []
excluded_subjects = []

print("--- Starting Batch Validation & Event Extraction ---")

# Iterate through every loaded subject
for idx, raw in enumerate(all_raws):
    # Subject IDs typically start at 1
    subject_id = idx + 1

    n_channels = raw.info["nchan"]
    sfreq = raw.info["sfreq"]

    # 1. Strict Validation: Check for 160 Hz rate and 64 channels
    if sfreq != 160.0 or n_channels != 64:
        excluded_subjects.append(subject_id)
        print(
            f"{colorama.Fore.RED}Subject {subject_id}: excluded due to invalid sampling rate or channel count."
        )
        continue  # Skip to the next subject without adding them to our valid lists

    # 2. Event Extraction
    try:
        # Extract events using our T0/T1/T2 mapping
        # Extract the events using the custom mapping
        # ? events is a numpy array where each row corresponds to an event, and the columns represent:
        # ?  the sample (index), (previous event - default to 0), (event code).
        # ? Event Dictionary Mapping: This confirms that your custom_mapping worked. I
        events, event_dict = mne.events_from_annotations(raw, event_id=custom_mapping)

        # Store the validated raw object and its events
        valid_raws.append(raw)
        all_events.append(events)

    except ValueError as e:
        # Catch errors if a subject's file is corrupted or missing expected annotations
        excluded_subjects.append(subject_id)
        print(f"{colorama.Fore.YELLOW}Subject {subject_id} excluded due to annotation error: {e}")

# ==========================================
# 3. Summary Report
# ==========================================
print(colorama.Fore.MAGENTA + "\n--- Processing Summary ---")
print(
    f"✅ Successfully prepared data for:{colorama.Style.BRIGHT}{colorama.Fore.GREEN} {len(valid_raws)}{colorama.Fore.RESET} {colorama.Style.NORMAL} subjects."
)
print(
    f"❌ Excluded{colorama.Style.BRIGHT}{colorama.Fore.RED} {len(excluded_subjects)}{colorama.Fore.RESET} {colorama.Style.NORMAL} subjects due to hardware/data inconsistencies."
)

if excluded_subjects:
    print(f"{colorama.Fore.RED}Excluded Subject IDs: {excluded_subjects}")

# Quick sanity check on the first valid subject's event array shape
if all_events:
    for idx, events in enumerate(all_events):
        print(f"Subject {idx + 1} has {len(events)} events.")

--- Starting Batch Validation & Event Extraction ---

--- Processing Summary ---
✅ Successfully prepared data for: 2  subjects.
❌ Excluded 0  subjects due to hardware/data inconsistencies.
Subject 1 has 362 events.
Subject 2 has 362 events.
